# Semana 9: Automatizando el Gradiente
## Notebook de Ejercicios (NB2) - Regresión Logística en MNIST con PyTorch

### Propósito de la sesión
Aplicar los conceptos de diferenciación automática y optimizadores a un problema real de clasificación de dígitos. Implementaremos una **regresión logística multiclase** (una capa lineal + softmax) en PyTorch, visualizaremos las curvas de pérdida y accuracy, y compararemos el rendimiento de diferentes optimizadores y tasas de aprendizaje.

### Objetivos de aprendizaje
*   Cargar y preprocesar el dataset **MNIST** para PyTorch.
*   Implementar un modelo de **regresión logística** (softmax) usando `nn.Linear`.
*   Entrenar el modelo usando diferentes **optimizadores** (SGD, Adam).
*   Visualizar la **pérdida** y la **accuracy** en entrenamiento y validación.
*   Experimentar con diferentes **learning rates** y analizar su efecto.
*   Comprender la importancia de la elección del optimizador en problemas reales.

## Configuración Inicial

Importamos las librerías necesarias: PyTorch para el modelo, torchvision para MNIST, y matplotlib para visualizaciones.

In [ ]:
# Importamos librerías
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
import time

# Configuración de matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Verificamos versión de PyTorch y disponibilidad de GPU
print(f"PyTorch versión: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")

# Fijamos semillas para reproducibilidad
torch.manual_seed(42)
np.random.seed(42)

print("\n🎯 Librerías importadas correctamente.")

---
## 1. Carga y Preprocesamiento de MNIST

MNIST contiene imágenes de 28x28 píxeles en escala de grises (1 canal). Aplanaremos las imágenes a vectores de 784 dimensiones para alimentar la capa lineal.

In [ ]:
# Transformaciones: convertir a tensor y normalizar a [0,1]
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # media y std de MNIST
])

# Descargar y cargar MNIST
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Crear dataloaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"🔷 MNIST cargado correctamente.")
print(f"  Train samples: {len(train_dataset)}")
print(f"  Test samples: {len(test_dataset)}")
print(f"  Batch size: {batch_size}")
print(f"  Número de batches entrenamiento: {len(train_loader)}")

In [ ]:
# Visualizamos algunas imágenes del dataset
def show_mnist_samples(loader, num_samples=8):
    data_iter = iter(loader)
    images, labels = next(data_iter)
    
    plt.figure(figsize=(12, 4))
    for i in range(num_samples):
        plt.subplot(2, 4, i+1)
        plt.imshow(images[i].squeeze(), cmap='gray')
        plt.title(f'Etiqueta: {labels[i].item()}')
        plt.axis('off')
    plt.suptitle('Muestras de MNIST')
    plt.tight_layout()
    plt.show()

show_mnist_samples(train_loader)

---
## 2. Definición del Modelo: Regresión Logística (Softmax)

La regresión logística multiclase consiste en una capa lineal que produce **logits** (puntuaciones para cada clase), seguida de softmax para obtener probabilidades.

$$\text{logits} = \mathbf{W} \mathbf{x} + \mathbf{b}$$
$$\hat{y} = \text{softmax}(\text{logits})$$

In [ ]:
class LogisticRegression(nn.Module):
    def __init__(self, input_dim=28*28, num_classes=10):
        super(LogisticRegression, self).__init__()
        self.linear = nn.Linear(input_dim, num_classes)
    
    def forward(self, x):
        # Aplanamos la imagen: de (batch, 1, 28, 28) a (batch, 784)
        x = x.view(x.size(0), -1)
        logits = self.linear(x)
        return logits  # No aplicamos softmax aquí porque CrossEntropyLoss lo incluye

# Instanciamos el modelo
model = LogisticRegression().to(device)
print(model)

# Contamos parámetros
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal de parámetros: {total_params:,}")

---
## 3. Función de Entrenamiento y Evaluación

Definimos funciones reutilizables para entrenar y evaluar el modelo con diferentes configuraciones.

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Entrena el modelo por una época."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Estadísticas
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc

def evaluate(model, test_loader, criterion, device):
    """Evalúa el modelo en el conjunto de prueba."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(test_loader.dataset)
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc

def train_model(model, train_loader, test_loader, criterion, optimizer, epochs, device, verbose=True):
    """Entrena el modelo y guarda historia de pérdida y accuracy."""
    train_losses = []
    train_accs = []
    test_losses = []
    test_accs = []
    
    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)
        
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        test_losses.append(test_loss)
        test_accs.append(test_acc)
        
        if verbose and (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1:2d}/{epochs} - "
                  f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% - "
                  f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")
    
    return train_losses, train_accs, test_losses, test_accs

---
## Actividad 1: Entrenamiento con SGD

Entrenamos el modelo usando el optimizador SGD con learning rate 0.01.

In [ ]:
# Configuración para SGD
model_sgd = LogisticRegression().to(device)
criterion = nn.CrossEntropyLoss()
optimizer_sgd = optim.SGD(model_sgd.parameters(), lr=0.01)

epochs = 30

print("🔷 Entrenando con SGD (lr=0.01)...")
start_time = time.time()
train_loss_sgd, train_acc_sgd, test_loss_sgd, test_acc_sgd = train_model(
    model_sgd, train_loader, test_loader, criterion, optimizer_sgd, epochs, device
)
sgd_time = time.time() - start_time
print(f"\n✅ Entrenamiento completado en {sgd_time:.2f} segundos.")
print(f"Mejor accuracy en test: {max(test_acc_sgd):.2f}%")

---
## Actividad 2: Entrenamiento con Adam

Ahora usamos el optimizador Adam con learning rate 0.001 (típico para Adam).

In [ ]:
# Configuración para Adam
model_adam = LogisticRegression().to(device)
optimizer_adam = optim.Adam(model_adam.parameters(), lr=0.001)

print("\n🔶 Entrenando con Adam (lr=0.001)...")
start_time = time.time()
train_loss_adam, train_acc_adam, test_loss_adam, test_acc_adam = train_model(
    model_adam, train_loader, test_loader, criterion, optimizer_adam, epochs, device
)
adam_time = time.time() - start_time
print(f"\n✅ Entrenamiento completado en {adam_time:.2f} segundos.")
print(f"Mejor accuracy en test: {max(test_acc_adam):.2f}%")

---
## Actividad 3: Visualización de pérdida y accuracy

Comparamos las curvas de aprendizaje de SGD y Adam.

In [ ]:
plt.figure(figsize=(14, 10))

# Pérdida en entrenamiento
plt.subplot(2, 2, 1)
plt.plot(train_loss_sgd, 'b-', linewidth=2, label='SGD (lr=0.01)')
plt.plot(train_loss_adam, 'r-', linewidth=2, label='Adam (lr=0.001)')
plt.xlabel('Época')
plt.ylabel('Pérdida (train)')
plt.title('Pérdida en entrenamiento')
plt.legend()
plt.grid(True)

# Pérdida en test
plt.subplot(2, 2, 2)
plt.plot(test_loss_sgd, 'b-', linewidth=2, label='SGD (lr=0.01)')
plt.plot(test_loss_adam, 'r-', linewidth=2, label='Adam (lr=0.001)')
plt.xlabel('Época')
plt.ylabel('Pérdida (test)')
plt.title('Pérdida en validación')
plt.legend()
plt.grid(True)

# Accuracy en entrenamiento
plt.subplot(2, 2, 3)
plt.plot(train_acc_sgd, 'b-', linewidth=2, label='SGD (lr=0.01)')
plt.plot(train_acc_adam, 'r-', linewidth=2, label='Adam (lr=0.001)')
plt.xlabel('Época')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy en entrenamiento')
plt.legend()
plt.grid(True)

# Accuracy en test
plt.subplot(2, 2, 4)
plt.plot(test_acc_sgd, 'b-', linewidth=2, label='SGD (lr=0.01)')
plt.plot(test_acc_adam, 'r-', linewidth=2, label='Adam (lr=0.001)')
plt.xlabel('Época')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy en validación')
plt.legend()
plt.grid(True)

plt.suptitle('Comparación SGD vs Adam en MNIST', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print(f"📊 Accuracy final - SGD: {test_acc_sgd[-1]:.2f}%, Adam: {test_acc_adam[-1]:.2f}%")

---
## Actividad 4: Experimentación con Learning Rates

Probamos diferentes learning rates para SGD y observamos su efecto.

In [ ]:
learning_rates = [0.001, 0.01, 0.1, 0.5]
epochs_lr = 15

plt.figure(figsize=(14, 8))

for i, lr in enumerate(learning_rates):
    model_lr = LogisticRegression().to(device)
    optimizer_lr = optim.SGD(model_lr.parameters(), lr=lr)
    
    train_loss, _, test_loss, test_acc = train_model(
        model_lr, train_loader, test_loader, criterion, optimizer_lr, epochs_lr, device, verbose=False
    )
    
    plt.subplot(2, 2, i+1)
    plt.plot(test_loss, linewidth=2, label=f'Test Loss')
    plt.plot(test_acc, linewidth=2, label=f'Test Acc')
    plt.xlabel('Época')
    plt.title(f'SGD con lr={lr}')
    plt.legend()
    plt.grid(True)

plt.suptitle('Efecto del Learning Rate en SGD', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Resumen de accuracy final para diferentes learning rates
print("📊 Accuracy final en test con diferentes learning rates (SGD):")
for lr in learning_rates:
    model_lr = LogisticRegression().to(device)
    optimizer_lr = optim.SGD(model_lr.parameters(), lr=lr)
    _, _, _, test_acc = train_model(
        model_lr, train_loader, test_loader, criterion, optimizer_lr, 10, device, verbose=False
    )
    print(f"  lr={lr}: {test_acc[-1]:.2f}%")

---
## Análisis de Resultados

*   **SGD con lr=0.01** converge de manera estable pero relativamente lenta.
*   **Adam con lr=0.001** converge más rápido y alcanza mayor accuracy en menos épocas.
*   **Learning rate muy bajo** (0.001) hace que SGD converja muy lentamente.
*   **Learning rate muy alto** (0.5) puede causar inestabilidad o divergencia.

Adam adapta la tasa de aprendizaje por parámetro, lo que lo hace más robusto a la elección del learning rate inicial.

---
## Ejercicios para el Estudiante

1.  **Momentum en SGD:** Prueba SGD con momentum (`optim.SGD(..., momentum=0.9)`). ¿Cómo afecta a la convergencia?

2.  **Optimizadores adicionales:** Experimenta con **RMSprop** y **Adagrad**. Compáralos con SGD y Adam.

3.  **Decaimiento de learning rate:** Implementa un scheduler que reduzca el learning rate cada cierto número de épocas (ej. `StepLR`). ¿Mejora el resultado?

4.  **Regularización:** Añade regularización L2 (weight decay) a los optimizadores. ¿Cómo afecta al overfitting?

5.  **Red más profunda:** Modifica el modelo para tener una capa oculta (ej. 256 neuronas con ReLU). Compara el rendimiento con la regresión logística simple.

6.  **Reflexión:** ¿Por qué Adam es generalmente preferido sobre SGD en problemas de deep learning? ¿En qué casos podría ser mejor SGD?

---
## Conclusiones de la Sesión

*   Implementamos una **regresión logística multiclase** en PyTorch para clasificar dígitos MNIST, usando una capa lineal y softmax.
*   Entrenamos el modelo con **SGD** y **Adam**, visualizando las curvas de pérdida y accuracy en entrenamiento y validación.
*   Observamos que **Adam** converge más rápido y alcanza mejor accuracy que SGD con learning rates típicos.
*   Experimentamos con diferentes **learning rates**, viendo su efecto en la velocidad de convergencia y estabilidad.
*   Comprendimos la importancia de elegir el optimizador adecuado y cómo los frameworks modernos (PyTorch) facilitan esta experimentación.
*   Este ejercicio consolida el uso de **diferenciación automática** y **optimizadores** en problemas reales de clasificación.

¡Ahora sabes entrenar modelos multiclase y optimizar su rendimiento con diferentes algoritmos!